<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/2_LangGraph_with_LangSmith_Support_Triage2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph + LangSmith — See It, Measure It, Fix It

**Notebook 2 of 2** · Same support-triage workflow, now with full observability.
**Runs entirely on free Google Colab. Open-source model on the Colab GPU. LangSmith free tier.**

### Before you run anything
In Colab: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

### The problem Notebook 1 left us with

Our workflow runs. But when it gets something wrong, we are blind:

| Question | Notebook 1 answer |
|---|---|
| Which node was slow? | print statements, maybe |
| What exact prompt did the model see? | you would have to log it yourself |
| Did it route this ticket correctly? | read every output by hand |
| Did my prompt change help or hurt? | vibes |
| How many tokens did that cost? | no idea |

**LangSmith answers all five.** Think of it as *"Google Analytics + a unit-test runner, for LLM apps."*

### The four things LangSmith does
| Capability | What it means | Section |
|---|---|---|
| **Tracing** | Records every node, prompt, output, latency, token count | 2–4 |
| **Debugging** | Click into any failed run and see exactly where it went wrong | 5 |
| **Evaluation** | Score your app against a dataset of known-correct examples | 6–8 |
| **Integration** | Two environment variables. No code changes. | 1 |

> **The one-sentence version:** LangGraph *runs* your workflow; LangSmith *shows you what it did* and *tells you whether it was right*.

---
## 1. Setup — the integration is two environment variables

1. Sign up free: <https://smith.langchain.com> (personal tier is free, no card).
2. **Settings → API Keys → Create API Key.** Copy it.
3. Run the cells below.

That is genuinely the whole integration. LangGraph and LangChain emit tracing callbacks automatically; setting `LANGSMITH_TRACING=true` turns them on.

In [ ]:
%pip install -q langgraph langchain-core langchain-huggingface transformers accelerate langsmith

In [ ]:
import os, getpass, warnings, torch

warnings.filterwarnings('ignore')            # keep the output clean for class
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_PROJECT'] = 'support-triage-class'   # groups runs; created on first trace
if not os.environ.get('LANGSMITH_API_KEY'):
    os.environ['LANGSMITH_API_KEY'] = getpass.getpass('LANGSMITH_API_KEY: ')

# If your LangSmith account is in the EU/APAC region, also set:
# os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'

print('Tracing on. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch to T4!')

> **Point to remember #1**
> The env vars must be set **before** you build the graph. Set them in the very first cell of any notebook.
>
> | Variable | Purpose |
> |---|---|
> | `LANGSMITH_TRACING` | `true` to turn tracing on |
> | `LANGSMITH_API_KEY` | your key |
> | `LANGSMITH_PROJECT` | folder name for grouping runs (optional but do it) |
> | `LANGSMITH_ENDPOINT` | only for EU / APAC / AWS regions |
>
> *(You may see older tutorials using `LANGCHAIN_TRACING_V2` and `LANGCHAIN_API_KEY`. Those are the legacy names for the same thing and still work.)*

---
## 2. Rebuild the workflow (condensed from Notebook 1)

Nothing here is new — same state, same nodes, same edges. Read it as a recap.
The important part: **we changed zero lines to enable tracing.**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'   # open weights, Apache 2.0, fits a free T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
text_gen = pipeline('text-generation', model=model, tokenizer=tokenizer,
                    max_new_tokens=256, do_sample=False, return_full_text=False,
                    pad_token_id=tokenizer.eos_token_id)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen))
print('Model loaded:', MODEL_ID)

In [ ]:
import json, re, operator
from typing import Annotated, TypedDict
from typing_extensions import Literal
from pydantic import BaseModel, ValidationError


class TicketState(TypedDict):
    ticket_id: str
    customer_tier: Literal['free', 'pro', 'enterprise']
    message: str
    category: str
    urgency: str
    draft_reply: str
    needs_human: bool
    reason: str
    log: Annotated[list[str], operator.add]


class Triage(BaseModel):
    category: Literal['billing', 'technical', 'general']
    urgency: Literal['low', 'high']

In [ ]:
from langsmith import traceable

TRIAGE_PROMPT = (
    'You are a support ticket classifier.\n'
    'Reply with ONLY a JSON object and nothing else, in exactly this shape:\n'
    '{"category": "billing", "urgency": "low"}\n'
    'category must be one of: billing, technical, general.\n'
    'urgency must be "high" if the customer is blocked, losing money, or angry; otherwise "low".\n\n'
    'Ticket: '
)


def extract_json(text: str) -> dict:
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if not match:
        raise ValueError('no JSON object found')
    return json.loads(match.group())


# @traceable makes this plain Python function show up as its own step in the trace.
# Use it on any non-LangChain helper you want visibility into.
@traceable(name='parse_triage_json')
def parse_triage(raw: str) -> Triage:
    try:
        return Triage(**extract_json(raw))
    except (ValueError, ValidationError, json.JSONDecodeError):
        return Triage(category='general', urgency='low')   # safe fallback


def classify_ticket(message: str) -> Triage:
    raw = llm.invoke(TRIAGE_PROMPT + message).content
    return parse_triage(raw)

> **Point to remember #2**
> LangChain and LangGraph objects are traced **automatically**. Your own plain functions are not — decorate them with **`@traceable`** to make them appear in the trace tree.

In [ ]:
def classify(state: TicketState) -> dict:
    r = classify_ticket(state['message'])
    return {'category': r.category, 'urgency': r.urgency,
            'log': [f'classified as {r.category}/{r.urgency}']}


def _write_reply(state: TicketState, persona: str) -> dict:
    prompt = (f'{persona}\n'
              f'Customer tier: {state["customer_tier"]}. Urgency: {state["urgency"]}.\n'
              f'Write a short, warm reply of at most 4 sentences. '
              f'Do not invent policy, refund amounts, or dates.\n\n'
              f'Ticket: {state["message"]}')
    return {'draft_reply': llm.invoke(prompt).content.strip(),
            'log': [f'drafted {state["category"]} reply']}


def billing_reply(s):    return _write_reply(s, 'You are a billing support specialist.')
def technical_reply(s):  return _write_reply(s, 'You are a technical support engineer. Suggest one concrete first troubleshooting step.')
def general_reply(s):    return _write_reply(s, 'You are a friendly customer success agent.')


def quality_check(state: TicketState) -> dict:
    risky = ['refund', 'cancel', 'legal', 'lawsuit', 'gdpr', 'data breach']
    text = (state['message'] + ' ' + state['draft_reply']).lower()
    if any(w in text for w in risky):
        return {'needs_human': True, 'reason': 'sensitive topic detected', 'log': ['flagged: sensitive topic']}
    if state['urgency'] == 'high' and state['customer_tier'] == 'enterprise':
        return {'needs_human': True, 'reason': 'high-urgency enterprise customer', 'log': ['flagged: enterprise SLA']}
    return {'needs_human': False, 'reason': 'safe to auto-send', 'log': ['passed quality check']}


def escalate(state: TicketState) -> dict:
    return {'log': [f'escalated to human queue ({state["reason"]})']}


def route_by_category(state):  return {'billing': 'billing_reply', 'technical': 'technical_reply'}.get(state['category'], 'general_reply')
def route_after_check(state):  return 'escalate' if state['needs_human'] else 'done'

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TicketState)
for name, fn in [('classify', classify), ('billing_reply', billing_reply),
                 ('technical_reply', technical_reply), ('general_reply', general_reply),
                 ('quality_check', quality_check), ('escalate', escalate)]:
    builder.add_node(name, fn)

builder.add_edge(START, 'classify')
for n in ['billing_reply', 'technical_reply', 'general_reply']:
    builder.add_edge(n, 'quality_check')
builder.add_edge('escalate', END)

builder.add_conditional_edges('classify', route_by_category,
    {'billing_reply': 'billing_reply', 'technical_reply': 'technical_reply', 'general_reply': 'general_reply'})
builder.add_conditional_edges('quality_check', route_after_check,
    {'escalate': 'escalate', 'done': END})

app = builder.compile()
print('Graph compiled - and every run from now on is traced.')

---
## 3. Your first trace

Run one ticket, then open <https://smith.langchain.com> → your project **support-triage-class**.

### What you will see
A **tree**, one row per step:

```
support-triage-class                          2.41s   412 tokens
└── LangGraph                                 2.41s
    ├── classify                              0.83s
    │   ├── ChatHuggingFace                   0.79s   ← click: exact prompt + raw output
    │   └── parse_triage_json                 0.00s   ← your @traceable function
    ├── billing_reply                         1.55s
    │   └── ChatHuggingFace                   1.55s
    └── quality_check                         0.00s   ← no LLM, therefore free and instant
```

### Vocabulary
| Term | Meaning |
|---|---|
| **Run** | One traced step (a node, an LLM call, a tool) |
| **Trace** | The whole tree for one `invoke()` |
| **Project** | A folder of traces (`LANGSMITH_PROJECT`) |
| **Latency** | Time per step — instantly shows your bottleneck |
| **Tokens** | Input/output token counts — your cost driver |

In [ ]:
ticket = {'ticket_id': 'T-2001', 'customer_tier': 'pro',
          'message': 'I was billed twice for July. Please check my invoice.', 'log': []}

result = app.invoke(ticket)
print(result['category'], '|', result['urgency'], '| human:', result['needs_human'])
print('\nNow open https://smith.langchain.com and click into the newest trace.')

---
## 4. Making traces searchable: tags, metadata, run names

By default every trace looks the same. In a week you will have thousands.
Attach **metadata** and **tags** so you can filter later: *"show me all enterprise tickets from prompt v2 that escalated."*

Pass them through `config` — no graph changes needed.

In [ ]:
tickets = [
    {'ticket_id': 'T-2002', 'customer_tier': 'free',
     'message': 'The mobile app crashes every time I open Reports.', 'log': []},
    {'ticket_id': 'T-2003', 'customer_tier': 'enterprise',
     'message': 'Our whole team cannot log in. Production is down. This is critical.', 'log': []},
    {'ticket_id': 'T-2004', 'customer_tier': 'pro',
     'message': 'Do you have a dark mode on the roadmap?', 'log': []},
]

for t in tickets:
    app.invoke(t, config={
        'run_name': f"triage-{t['ticket_id']}",             # readable name in the UI
        'tags': ['prompt-v1', t['customer_tier']],          # filter chips
        'metadata': {'ticket_id': t['ticket_id'],           # searchable key/values
                     'tier': t['customer_tier'],
                     'model': MODEL_ID},
    })

print('4 traces sent. In the UI try the filter:  metadata.tier = enterprise')

> **Point to remember #3**
> Tag every run with the things you will want to slice by later: prompt version, model name, customer segment, environment. Untagged traces are almost impossible to analyse at volume.

---
## 5. Debugging with LangSmith — the actual workflow

When a ticket is handled badly, this is the loop:

| Step | Action |
|---|---|
| 1 | **Filter** the project: `Error = true`, or `metadata.tier = enterprise`, or latency `> 5s` |
| 2 | **Open the trace** and find the node where the output first looks wrong |
| 3 | **Read the Input tab** — the *exact* prompt string the model received. 80% of bugs are visible here (a missing variable, a stray `{}`, truncated context) |
| 4 | **Read the Output tab** — what the model actually returned, before your parsing |
| 5 | **Open in Playground** — edit the prompt and re-run that single step, without re-running the whole graph |
| 6 | **Add to dataset** — one click turns this bad run into a permanent regression test |

Step 6 is the habit that compounds. Every bug you find becomes a test you can never regress on.

### The three-question debugging checklist
1. *Did the node receive what I expected?* → **Input** tab
2. *Did the model return what I expected?* → **Output** tab
3. *Did my code parse it correctly?* → the `@traceable` step right after the LLM call

Below we deliberately break the prompt so you can see a failure in the trace.

In [ ]:
# A deliberately terrible prompt - vague, no format instruction.
BAD_PROMPT = 'What kind of ticket is this? '

def classify_badly(state: TicketState) -> dict:
    raw = llm.invoke(BAD_PROMPT + state['message']).content
    r = parse_triage(raw)          # this will hit the fallback path
    return {'category': r.category, 'urgency': r.urgency, 'log': ['classified (bad prompt)']}


bad_builder = StateGraph(TicketState)
for name, fn in [('classify', classify_badly), ('billing_reply', billing_reply),
                 ('technical_reply', technical_reply), ('general_reply', general_reply),
                 ('quality_check', quality_check), ('escalate', escalate)]:
    bad_builder.add_node(name, fn)
bad_builder.add_edge(START, 'classify')
for n in ['billing_reply', 'technical_reply', 'general_reply']:
    bad_builder.add_edge(n, 'quality_check')
bad_builder.add_edge('escalate', END)
bad_builder.add_conditional_edges('classify', route_by_category,
    {'billing_reply': 'billing_reply', 'technical_reply': 'technical_reply', 'general_reply': 'general_reply'})
bad_builder.add_conditional_edges('quality_check', route_after_check, {'escalate': 'escalate', 'done': END})
bad_app = bad_builder.compile()

out = bad_app.invoke(tickets[0], config={'run_name': 'BROKEN-classifier', 'tags': ['prompt-v0-broken']})
print('category returned:', out['category'], '(likely the fallback "general" - the crash was contained)')
print('\nOpen the BROKEN-classifier trace: the ChatHuggingFace output will be prose, not JSON.')

> **Point to remember #4**
> A trace shows you *what the model was actually asked*, not what you think you asked it. That single fact resolves most "the LLM is dumb" bug reports.

---
## 6. Evaluation — from "looks fine" to a number

Debugging fixes one ticket. **Evaluation tells you whether the system is good overall**, and whether your last change helped or hurt.

Three pieces:

| Piece | What it is |
|---|---|
| **Dataset** | A list of examples: `inputs` (a ticket) + `outputs` (the correct answer) |
| **Evaluator** | A function that scores one actual output against the reference |
| **Experiment** | One run of your app over the whole dataset, scored |

### Where do datasets come from?
1. Hand-written, like below — 10 to 20 examples is enough to start.
2. **Clicked in from real traces** — the highest-value source. Every production mistake becomes a test case.
3. Exported from your existing support tickets.

> Do not wait for a perfect dataset. **20 examples you actually run beats 500 you never build.**

In [ ]:
from langsmith import Client

client = Client()

examples = [
    {'inputs': {'ticket_id': 'E1', 'customer_tier': 'pro',
                'message': 'I was charged twice for my subscription this month.'},
     'outputs': {'category': 'billing', 'needs_human': False}},

    {'inputs': {'ticket_id': 'E2', 'customer_tier': 'free',
                'message': 'The export button does nothing when I click it.'},
     'outputs': {'category': 'technical', 'needs_human': False}},

    {'inputs': {'ticket_id': 'E3', 'customer_tier': 'enterprise',
                'message': 'Production is completely down, none of our 200 users can log in.'},
     'outputs': {'category': 'technical', 'needs_human': True}},

    {'inputs': {'ticket_id': 'E4', 'customer_tier': 'pro',
                'message': 'I want a refund for last month, the service was unusable.'},
     'outputs': {'category': 'billing', 'needs_human': True}},

    {'inputs': {'ticket_id': 'E5', 'customer_tier': 'free',
                'message': 'Where can I find your API documentation?'},
     'outputs': {'category': 'general', 'needs_human': False}},

    {'inputs': {'ticket_id': 'E6', 'customer_tier': 'enterprise',
                'message': 'Please update the billing address on our account.'},
     'outputs': {'category': 'billing', 'needs_human': False}},

    {'inputs': {'ticket_id': 'E7', 'customer_tier': 'pro',
                'message': 'Your app deleted my data. I am contacting my lawyer.'},
     'outputs': {'category': 'technical', 'needs_human': True}},

    {'inputs': {'ticket_id': 'E8', 'customer_tier': 'free',
                'message': 'Do you offer a student discount?'},
     'outputs': {'category': 'general', 'needs_human': False}},
]

DATASET_NAME = 'support-triage-golden-v1'

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print('Dataset already exists - reusing it.')
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description='Hand-labelled support tickets: correct category + whether a human is needed.')
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print('Created dataset with', len(examples), 'examples.')

---
## 7. Evaluators — turning judgement into a score

An evaluator is a **plain Python function**. LangSmith inspects its argument *names* and passes what you ask for:

| Argument | What you get |
|---|---|
| `inputs` | the example's input dict |
| `outputs` | what your app actually returned |
| `reference_outputs` | the correct answer from the dataset |

Return a `bool`, a number, or a `{'key': ..., 'score': ...}` dict.

### Two families of evaluator

**Rule-based** — exact, free, instant. Use whenever there is a right answer.
**LLM-as-judge** — for things with no exact answer (tone, helpfulness, groundedness). Slower, imperfect, but far better than nothing.

Start with rule-based. Reach for LLM-as-judge only when a rule genuinely cannot express the quality you care about.

In [ ]:
# ---------- Rule-based evaluators ----------

# Did we route the ticket to the right team?
def correct_category(outputs: dict, reference_outputs: dict) -> bool:
    return outputs.get('category') == reference_outputs.get('category')


# Did we correctly decide whether a human is needed?
def correct_escalation(outputs: dict, reference_outputs: dict) -> bool:
    return outputs.get('needs_human') == reference_outputs.get('needs_human')


# Cheap sanity guard: did we actually produce a reply?
def reply_not_empty(outputs: dict) -> bool:
    return len(outputs.get('draft_reply', '').strip()) > 20

In [ ]:
# ---------- LLM-as-judge evaluator ----------
# Judges tone, which no rule can capture. Note it returns a 0/1 score with a key.

JUDGE_PROMPT = (
    'You are grading a customer support reply.\n'
    'Answer with ONLY one word: GOOD or BAD.\n'
    'Answer GOOD if the reply is polite, on-topic, and makes no invented promises '
    '(no specific refund amounts, dates, or policies).\n'
    'Answer BAD otherwise.\n\n'
)

def polite_and_grounded(inputs: dict, outputs: dict) -> dict:
    verdict = llm.invoke(
        JUDGE_PROMPT + f"Ticket: {inputs['message']}\n\nReply: {outputs.get('draft_reply', '')}"
    ).content.upper()
    return {'key': 'polite_and_grounded', 'score': 1 if 'GOOD' in verdict else 0}

> **Point to remember #5**
> Evaluator arguments are matched **by name**. `outputs`, `reference_outputs`, `inputs` are the magic names — a typo means your evaluator silently gets nothing.
>
> **Point to remember #6**
> An LLM judge is itself an LLM app, with all the same flaws. Spot-check its verdicts before you trust its numbers.

---
## 8. Running the experiment

`evaluate()` needs a **target function**: takes one `inputs` dict, returns one `outputs` dict.

Ours just adds the `log` key the graph expects, invokes the graph, and returns the fields we grade.

In [ ]:
def target(inputs: dict) -> dict:
    final = app.invoke({**inputs, 'log': []})
    return {
        'category': final['category'],
        'needs_human': final['needs_human'],
        'draft_reply': final['draft_reply'],
    }


results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[correct_category, correct_escalation, reply_not_empty, polite_and_grounded],
    experiment_prefix='qwen2.5-3b-prompt-v1',
    description='Baseline: structured JSON classify prompt.',
    max_concurrency=1,          # 1 for a single local GPU; raise it for hosted APIs
    metadata={'model': MODEL_ID, 'prompt_version': 'v1'},
)

print('Done. Open the experiment link printed above.')

In [ ]:
# Read the scores back in code (useful for CI pipelines)
import collections

scores = collections.defaultdict(list)
for row in results:
    for fb in row['evaluation_results']['results']:
        scores[fb.key].append(float(fb.score))

for key, vals in scores.items():
    print(f'{key:<22} {sum(vals)/len(vals):.0%}  ({int(sum(vals))}/{len(vals)})')

---
## 9. Comparing two versions — the payoff

The real value is **regression testing**: change one thing, re-run, compare.

Below we change the classify prompt to add a worked example (one-shot prompting), run a second experiment on the *same* dataset, then compare them side by side in the UI.

**In LangSmith:** open the dataset → **Experiments** tab → tick both → **Compare**.
Green cells improved, red cells regressed. You can see the *exact examples* that changed behaviour.

In [ ]:
# Prompt v2: same instructions, plus one worked example
TRIAGE_PROMPT_V2 = (
    'You are a support ticket classifier.\n'
    'Reply with ONLY a JSON object and nothing else, in exactly this shape:\n'
    '{"category": "billing", "urgency": "low"}\n'
    'category must be one of: billing, technical, general.\n'
    'urgency must be "high" if the customer is blocked, losing money, or angry; otherwise "low".\n\n'
    'Example:\n'
    'Ticket: My card was declined and I lost access to the dashboard.\n'
    '{"category": "billing", "urgency": "high"}\n\n'
    'Ticket: '
)


def classify_v2(state: TicketState) -> dict:
    r = parse_triage(llm.invoke(TRIAGE_PROMPT_V2 + state['message']).content)
    return {'category': r.category, 'urgency': r.urgency, 'log': ['classified v2']}


v2 = StateGraph(TicketState)
for name, fn in [('classify', classify_v2), ('billing_reply', billing_reply),
                 ('technical_reply', technical_reply), ('general_reply', general_reply),
                 ('quality_check', quality_check), ('escalate', escalate)]:
    v2.add_node(name, fn)
v2.add_edge(START, 'classify')
for n in ['billing_reply', 'technical_reply', 'general_reply']:
    v2.add_edge(n, 'quality_check')
v2.add_edge('escalate', END)
v2.add_conditional_edges('classify', route_by_category,
    {'billing_reply': 'billing_reply', 'technical_reply': 'technical_reply', 'general_reply': 'general_reply'})
v2.add_conditional_edges('quality_check', route_after_check, {'escalate': 'escalate', 'done': END})
app_v2 = v2.compile()


print('v2 graph compiled.')

In [ ]:
def target_v2(inputs: dict) -> dict:
    final = app_v2.invoke({**inputs, 'log': []})
    return {'category': final['category'], 'needs_human': final['needs_human'],
            'draft_reply': final['draft_reply']}


results_v2 = client.evaluate(
    target_v2,
    data=DATASET_NAME,
    evaluators=[correct_category, correct_escalation, reply_not_empty, polite_and_grounded],
    experiment_prefix='qwen2.5-3b-prompt-v2',
    description='Added a one-shot example to the classify prompt.',
    max_concurrency=1,
    metadata={'model': MODEL_ID, 'prompt_version': 'v2'},
)

print('Now compare v1 vs v2 in the LangSmith UI: dataset -> Experiments -> select both -> Compare.')

> **Point to remember #7**
> Never judge a prompt change by eyeballing one output. Re-run the dataset. Small models are noisy — a change that "obviously helps" often does not.

---
## 10. Beyond the notebook — what teams actually do

| Practice | Why |
|---|---|
| **Separate projects per environment** | `LANGSMITH_PROJECT=triage-dev` / `triage-prod` — keep experiments out of prod dashboards |
| **Add failing prod traces to the dataset** | One click. Your test suite grows from real failures |
| **Run `evaluate()` in CI** | Block the merge if `correct_category` drops below 85% |
| **Online evaluators** | Configure a rule in the UI to auto-score a sample of live traffic |
| **Human annotation queues** | Route uncertain runs to a human reviewer, feed labels back to the dataset |
| **Monitoring dashboards** | Alert on latency spikes, error rates, token/cost creep |
| **`@traceable` on helper code** | Traces are only as useful as the steps you make visible |

### Cost and privacy notes for the workplace
- The free personal tier includes a generous monthly trace allowance — plenty for learning.
- Traces contain your **prompts and outputs**. Before pointing this at real customer data, check your company's policy. LangSmith supports self-hosting and payload masking for exactly this reason.

---
## Cheat sheet — screenshot this

### Setup
```python
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = '...'
os.environ['LANGSMITH_PROJECT'] = 'my-project'
```

### Tracing
| Task | Code |
|---|---|
| Trace LangChain/LangGraph | automatic |
| Trace your own function | `@traceable` decorator |
| Name a run | `config={'run_name': 'triage-T-1001'}` |
| Tag a run | `config={'tags': ['prompt-v1']}` |
| Attach searchable data | `config={'metadata': {'tier': 'pro'}}` |

### Evaluation
| Task | Code |
|---|---|
| Client | `client = Client()` |
| Create dataset | `client.create_dataset(dataset_name=...)` |
| Add examples | `client.create_examples(dataset_id=..., examples=[...])` |
| Evaluator signature | `def f(inputs, outputs, reference_outputs) -> bool` |
| LLM-judge return | `{'key': 'name', 'score': 0 or 1}` |
| Run experiment | `client.evaluate(target, data=..., evaluators=[...])` |
| Name the experiment | `experiment_prefix='v1'` |
| Speed up | `max_concurrency=4` (hosted APIs only) |

### The 7 things to remember

1. **Two env vars = full tracing.** Set them before building the graph.
2. **`@traceable`** makes your own functions visible in the trace.
3. **Tag and add metadata to every run** — future you needs to filter.
4. **Read the Input tab first.** Most bugs are a malformed prompt, not a dumb model.
5. **Turn every production failure into a dataset example.**
6. **Rule-based evaluators first**, LLM-as-judge only for things rules cannot express.
7. **Compare experiments, never single outputs.** That is how you know a change was real.

### Common errors

| Symptom | Cause | Fix |
|---|---|---|
| No traces appear | Env vars set after the graph was built | Set them in cell 1, restart runtime |
| `401 Unauthorized` | Wrong key, or wrong region | Re-copy the key; set `LANGSMITH_ENDPOINT` for EU/APAC |
| Evaluator always scores 0 | Argument name typo | Must be exactly `inputs`, `outputs`, `reference_outputs` |
| `KeyError` in evaluator | Target returned different keys than the dataset uses | Print `outputs` once and align the key names |
| Evaluation extremely slow | One local GPU serialising every call | Keep `max_concurrency=1` locally; it is expected |
| Dataset already exists | Re-ran the cell | Handled by the `has_dataset` check |

---
## Exercises

1. **Easy** — add an `urgency` label to the dataset and write a `correct_urgency` evaluator.
2. **Easy** — tag your runs with `environment: dev` and filter for them in the UI.
3. **Medium** — write an evaluator that fails any reply containing a currency amount (a hallucinated-refund guard).
4. **Medium** — add three tickets the current system gets *wrong* to the dataset, then fix the prompt until they pass.
5. **Hard** — swap `MODEL_ID` to `Qwen/Qwen2.5-0.5B-Instruct`, re-run the same experiment, and compare accuracy against latency. Is the bigger model worth it for *your* use case?

---
### The takeaway

> **LangGraph gives you control over how your LLM app runs.**
> **LangSmith gives you evidence about whether it works.**
>
> You need both. Control without evidence is guessing; evidence without control is a dashboard you cannot act on.